In [ ]:
using ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi*(rand(1)[1]/5+0.90)
ω_b_0 = 2*ω_a# 7.0515*2*pi*(rand(1)[1]/5+0.90)


φ_a = 0.1123
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 36.924*2*pi#*(rand(1)[1]/4+0.875)


##Drive

ω_d=ω_b_0
ω_p=2ω_a-ω_d

ϵ_0 = 0.4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.124276*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max = 30/κ_b
t_min = 0.0
t_span = (t_min, t_max)
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

In [ ]:
ϵ_0_list = vcat(0.5:.011:1,1:.15:0.6pi);
#ϵ_0_list = vcat(0.95:.005/3:1.05);

In [ ]:
FFT_array = Array{Vector{Float64}}(undef, length(ϵ_0_list))
Shannon_entropy_array = Array{Float64}(undef, length(ϵ_0_list))
freqs = Vector{Float64}()
drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
for i in ProgressBar(1:length(ϵ_0_list))
    ϵ_0_i = ϵ_0_list[i]
    drive.ϵ_0 = ϵ_0_i
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    
    ϵ_d_i = 10.124276*g2_max_i
    drive.ϵ_d = ϵ_d_i
    drive_prepared.drive = drive
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

    ϕ_a, _, _, _, t_list = default_solver((u_dot_0, u_0), t_span, p, dt; equation=default_equation!)
    if i == length(ϵ_0_list)
        F, freqs = FFT_perso(ϕ_a, t_list)
    else
        F, _ = FFT_perso(ϕ_a, t_list)
    end
    FFT_array[i] = abs.(F)
    power_density = abs.(F).^2
    power_density_normalized = power_density ./sum(power_density)
    Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))
    Shannon_entropy_array[i] = Shannon_entropy
end    

In [ ]:
FFT_matrix = reduce(hcat, FFT_array);

In [ ]:
mask = (freqs .>= 0) .& (freqs .<= 15)
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b
heatmap(g2_list*1e3/2pi, freqs[mask], log10.(FFT_matrix[mask, :]), cmap = Perso_colormap)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel="Frequency (GHz)",
title = "FFT of the phase of the memory initialized in vacuum", colorbar=:none)
plot!(twinx(), g2_list*1e3/2pi,Shannon_entropy_array, ylabel = "Shannon entropy", label="")
#colorbar_title="\n"*L"\log_{10}(|F|)")
#savefig("../../../figures/chaos/heatmap_FFT_vacuum_shannon_entropy_pump_12_12_2025.pdf")

In [ ]:
mask = (freqs .>= 0) .& (freqs .<= 15)
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b
heatmap(ϵ_0_list, freqs[mask], log10.(FFT_matrix[mask, :]), cmap = Perso_colormap)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel="Frequency (GHz)",
title = "FFT of the phase of the memory initialized in vaccuum", colorbar=:none)
plot!(twinx(), ϵ_0_list,Shannon_entropy_array)
#colorbar_title="\n"*L"\log_{10}(|F|)")
#savefig("../../../figures/chaos/heatmap_FFT_vacuum.pdf")

In [ ]:
Shannon_entropy_array = Array{Float64}(undef, length(ϵ_0_list))
n=20
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)
for i in 1:n
    ω_a = 3.8533*2*pi*(rand(1)[1]/5+0.90)
    ω_b_0 = 2*ω_a# 7.0515*2*pi*(rand(1)[1]/5+0.90)

    C_a_inv = Z_a*ω_a
    E_Ca = C_a_inv/2

    C_b_inv = Z_b*ω_b_0
    E_Cb = C_b_inv/2

    ##Drive
    ω_d=ω_b_0
    ω_p=2ω_a-ω_d

    system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
    system_prepared = prepare(system)
    environment_prepared = prepare(system,environment)

    dt = get_dt(system, environment)
    u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
    
    drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    for i in 1:length(ϵ_0_list)
        ϵ_0_i = ϵ_0_list[i]
        drive.ϵ_0 = ϵ_0_i
        g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
        
        ϵ_d_i = 10.124276*g2_max_i
        drive.ϵ_d = ϵ_d_i
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        ϕ_a, _, _, _, t_list = default_solver((u_dot_0, u_0), t_span, p, dt; equation=default_equation!)
        
        F, _ = FFT_perso(ϕ_a, t_list)
        
        power_density = abs.(F).^2
        power_density_normalized = power_density ./sum(power_density)
        Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))
        Shannon_entropy_array[i] +=Shannon_entropy
        if i ==1
            println(Shannon_entropy)
        end
        if i ==length(ϵ_0_list)
            println(Shannon_entropy)
        end
    end    
end
Shannon_entropy_array = Shannon_entropy_array./n;

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b
plot(g2_list*1e3/2pi,Shannon_entropy_array, label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel="Shannon entropy")